# RSA Factorization — CSV-driven Batch + Results Cache

- Provide targets in `/mnt/data/rsa_targets.csv` with a header `N`.
- Results cached to `/mnt/data/rsa_results.json` so reruns are instant.

In [ ]:
import csv, json, os, time
from pathlib import Path
from quantum_hybrid_system import QuantumClassicalHybrid

csv_path = Path("/mnt/data/rsa_targets.csv")
cache_path = Path("/mnt/data/rsa_results.json")

# Create a template CSV if missing
if not csv_path.exists():
    with csv_path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["N"])
        w.writerows([[221],[299],[391],[437],[667],[713],[899],[1147],[1763],[2491]])
    print("Template written to", csv_path)

# Load cache
cache = {}
if cache_path.exists():
    try:
        cache = json.loads(cache_path.read_text())
    except Exception as e:
        print("Cache read error:", e)

targets = []
with csv_path.open() as f:
    rdr = csv.DictReader(f)
    for row in rdr:
        try:
            N = int(row["N"])
            targets.append(N)
        except:
            pass

hybrid = QuantumClassicalHybrid(verbose=True, use_gpu=True)
results = []

for N in targets:
    key = str(N)
    if key in cache:
        rec = cache[key]
        rec["_source"] = "cache"
        results.append(rec)
        print(f"N={N} from cache -> {rec['factors']}  {rec['ms']:.2f} ms")
        continue
    t0 = time.perf_counter()
    f = hybrid.factor_number(N)
    dt = (time.perf_counter() - t0) * 1e3
    rec = {"N": N, "factors": f, "ms": dt, "_source": "fresh"}
    cache[key] = {"N": N, "factors": f, "ms": dt}
    results.append(rec)
    print(f"N={N} fresh -> {f}  {dt:.2f} ms")

# Save cache
cache_path.write_text(json.dumps(cache, indent=2))
print("Cache updated at", cache_path)

# Pretty print summary
print("\\nSummary:")
for r in results:
    print(f"{r['N']:<6} {r['factors']}  {r['ms']:.2f} ms  ({r['_source']})")